# MIRT: Multiclass Informative Resampling Technique

Full replication of the experiment from:
> *Development of a Preprocessing Technique for Multiclass Imbalanced Dataset Based on Data Similarity Degree and Data Difficulty Factors (MIRT)*

## Overview
MIRT handles multiclass imbalanced datasets by:
1. **Identifying example types** — safe, borderline, rare, outlier via 5-NN neighborhood analysis
2. **Computing inter-class similarity degrees** — based on attribute mean differences between classes
3. **Evaluating safe levels** — combining similarity and neighborhood class distribution
4. **Informative resampling** — oversampling minority (high safe-level) + undersampling majority (high safe-level)
5. **Classification & Evaluation** — KNN, SVM, CART with Accuracy, F1-score, G-Mean, AUC ROC

**Datasets:** new-thyroid (NT), hayes-roth (HR), balance-scale (BS), car (CA), cmc (CMC)

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import math
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm, metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, roc_auc_score,
    f1_score, classification_report
)

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

try:
    from imblearn.metrics import geometric_mean_score
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imbalanced-learn', '-q'])
    from imblearn.metrics import geometric_mean_score

print('All imports successful.')

## 2. Dataset Loading

Five multiclass imbalanced datasets from the UCI repository are loaded and normalised:
- Numeric feature matrix `X`
- Integer class labels `y` (starting at 1, majority class = 1)

In [ ]:
BASE = r'C:\Users\USER\Desktop\ML Project\dataset\dataset\used'


def load_new_thyroid():
    """215 samples · 3 classes (150:35:30) · 5 continuous features.
    Format: Class, T3Resin, T4, T3, TSH, MaxTSH  (6 cols, no missing values)
    """
    feat = ['T3Resin', 'T4', 'T3', 'TSH', 'MaxTSH']
    df = pd.read_csv(f'{BASE}\\new-thyroid.data', header=None, names=['Class'] + feat)
    X = df[feat].values.astype(float)
    y = df['Class'].values.astype(int)
    return X, y, feat


def load_hayes_roth():
    """132 samples · 3 classes (51:51:30) · 4 integer features.
    Format: SampleID, f1, f2, f3, f4, Class  (first col is ID, last is class)
    """
    feat = ['f1', 'f2', 'f3', 'f4']
    df = pd.read_csv(f'{BASE}\\hayes-roth.data', header=None,
                     names=['id'] + feat + ['Class'])
    X = df[feat].values.astype(float)
    y = df['Class'].values.astype(int)
    return X, y, feat


def load_balance_scale():
    """625 samples · 3 classes (B=49, L=288, R=288) · 4 numeric features.
    B (balanced) is the minority.  Mapped: B→1 (minority), L→2, R→3.
    """
    feat = ['LW', 'LD', 'RW', 'RD']
    df = pd.read_csv(f'{BASE}\\balance-scale.data', header=None,
                     names=['Class'] + feat)
    # Map so minority (B) becomes class 1
    cls_map = {'B': 1, 'L': 2, 'R': 3}
    df['Class'] = df['Class'].map(cls_map)
    X = df[feat].values.astype(float)
    y = df['Class'].values.astype(int)
    return X, y, feat


def load_car():
    """1728 samples · 4 classes · 6 categorical features (ordinal-encoded).
    Classes by frequency: unacc(1210)→1, acc(384)→2, good(69)→3, vgood(65)→4.
    """
    feat = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety']
    df = pd.read_csv(f'{BASE}\\car.data', header=None, names=feat + ['Class'])

    # Ordinal encoding for each feature
    ordinals = {
        'buying':   ['low', 'med', 'high', 'vhigh'],
        'maint':    ['low', 'med', 'high', 'vhigh'],
        'doors':    ['2', '3', '4', '5more'],
        'persons':  ['2', '4', 'more'],
        'lug_boot': ['small', 'med', 'big'],
        'safety':   ['low', 'med', 'high'],
    }
    for col, order in ordinals.items():
        df[col] = df[col].map({v: i for i, v in enumerate(order)})

    # Class: rank by frequency (most frequent = 1 = majority)
    freq_order = df['Class'].value_counts().index.tolist()
    cls_map = {cls: rank + 1 for rank, cls in enumerate(freq_order)}
    df['Class'] = df['Class'].map(cls_map)

    X = df[feat].values.astype(float)
    y = df['Class'].values.astype(int)
    return X, y, feat


def load_cmc():
    """1473 samples · 3 classes (no-use=629, long=511, short=333) · 9 features.
    Format: id, age, wife_edu, husb_edu, children, wife_rel, wife_work,
            husb_occ, sol, media, Class  (tab-separated, first col is ID)
    """
    feat = ['age', 'wife_edu', 'husb_edu', 'children', 'wife_rel',
            'wife_work', 'husb_occ', 'sol', 'media']
    cols = ['id'] + feat + ['Class']
    df = pd.read_csv(f'{BASE}\\cmc.data', sep='\t', header=None, names=cols)
    X = df[feat].values.astype(float)
    y = df['Class'].values.astype(int)
    return X, y, feat


DATASETS = {
    'new-thyroid (NT)':  load_new_thyroid,
    'hayes-roth (HR)':   load_hayes_roth,
    'balance-scale (BS)': load_balance_scale,
    'car (CA)':          load_car,
    'cmc (CMC)':         load_cmc,
}

print('Dataset summary:')
print(f'{"Dataset":<25} {"Samples":>8} {"Features":>9} {"Classes":>8}  Distribution')
print('-' * 80)
for name, loader in DATASETS.items():
    X, y, feat = loader()
    dist = dict(pd.Series(y).value_counts().sort_index())
    nan_count = np.isnan(X).sum()
    print(f'{name:<25} {len(y):>8} {len(feat):>9} {len(dist):>8}  {dist}  NaN={nan_count}')

## 3. Core MIRT Functions

### 3.1  Example Type Classification (Safe / Borderline / Rare / Outlier)

For each example, examine its 5 nearest neighbours:

| Same-class neighbours out of 5 | Type |
|---|---|
| 5 or 4 | **Safe (S)** |
| 3 or 2 | **Borderline (B)** |
| 1      | **Rare (R)** |
| 0      | **Outlier (O)** |

In [ ]:
def classify_type(same_count):
    if same_count >= 4:
        return 'S'
    elif same_count >= 2:
        return 'B'
    elif same_count == 1:
        return 'R'
    return 'O'


def get_5nn_info(X, y):
    """Fit 6-NN (to exclude self at index 0) and collect neighbourhood info."""
    neigh = KNeighborsClassifier(n_neighbors=6)
    neigh.fit(X, y)
    distances, indices = neigh.kneighbors(X, return_distance=True)

    records = []
    for i in range(len(y)):
        nn_idx  = indices[i, 1:6]      # skip self (position 0)
        nn_dist = distances[i, 1:6]
        nn_cls  = y[nn_idx]
        same    = int(np.sum(nn_cls == y[i]))
        records.append({
            'real_index':    i,
            'Class':         int(y[i]),
            'same_count':    same,
            'distance':      float(np.sum(nn_dist)),
            'type':          classify_type(same),
            'neigh_classes': nn_cls.tolist(),
        })
    return pd.DataFrame(records)


def print_type_summary(nn_df, classes):
    print(f'  {"Class":<8} {"Safe":>6} {"Border":>8} {"Rare":>6} {"Outlier":>9} {"Total":>7}')
    for cls in sorted(classes):
        sub = nn_df[nn_df['Class'] == cls]
        counts = sub['type'].value_counts()
        s = counts.get('S', 0)
        b = counts.get('B', 0)
        r = counts.get('R', 0)
        o = counts.get('O', 0)
        print(f'  {cls:<8} {s:>6} {b:>8} {r:>6} {o:>9} {len(sub):>7}')

### 3.2  Inter-Class Similarity Degree

For each pair of classes $(C_i, C_j)$:
1. Randomly sample 20 examples from each class
2. Average the per-attribute means → one representative value $\bar{v}$ per class
3. $\mu_{ij} = \text{clip}\left(1 - \frac{|\bar{v}_i - \bar{v}_j|}{10},\; 0,\; 1\right)$,\quad $\mu_{ii} = 1$

In [ ]:
def compute_class_value(X_cls, n_sample=20, rng=None):
    n = min(n_sample, len(X_cls))
    if rng is None:
        rng = np.random.default_rng()
    idx    = rng.choice(len(X_cls), size=n, replace=False)
    sample = X_cls[idx]
    return float(np.mean(np.mean(sample, axis=0)))


def compute_similarity_matrix(X, y, n_sample=20, seed=42):
    rng     = np.random.default_rng(seed)
    classes = sorted(np.unique(y).tolist())
    vals    = {cls: compute_class_value(X[y == cls], n_sample, rng) for cls in classes}

    mu = {}
    for ci in classes:
        mu[ci] = {}
        for cj in classes:
            if ci == cj:
                mu[ci][cj] = 1.0
            else:
                diff = abs(vals[ci] - vals[cj])
                mu[ci][cj] = float(np.clip(1.0 - diff / 10.0, 0.0, 1.0))
    return mu, vals


def print_similarity_matrix(mu, classes):
    header = '       ' + ''.join(f'{c:>8}' for c in classes)
    print(header)
    for ci in classes:
        row = f'  {ci:>4}  ' + ''.join(f'{mu[ci][cj]:>8.4f}' for cj in classes)
        print(row)

### 3.3  Safe Level

For each example $x$ from class $C_i$:
$$sl(x) = \frac{\sum_j \mu_{ij} \cdot n_{C_j}}{5}$$
where $n_{C_j}$ = number of the 5 neighbours belonging to class $C_j$.

In [ ]:
def compute_safe_levels(nn_df, mu, classes):
    sls = []
    for _, row in nn_df.iterrows():
        ci = int(row['Class'])
        nc = row['neigh_classes']
        sl = sum(mu[ci][cj] * nc.count(cj) for cj in classes) / 5.0
        sls.append(sl)
    result = nn_df.copy()
    result['safe_level'] = sls
    return result

### 3.4  MIRT Resampling

**Target size** $m = \lceil \text{mean of all class sizes} \rceil$

- **Majority class** → undersample to $m$, prioritising the safest examples
- **Minority classes** → oversample to $m$ with replacement, weighted by `safe_level`

In [ ]:
def mirt_resample(X, y, nn_df, mu, seed=42):
    rng     = np.random.default_rng(seed)
    classes = sorted(np.unique(y).tolist())
    sizes   = {c: int(np.sum(y == c)) for c in classes}
    m       = math.ceil(np.mean(list(sizes.values())))
    maj_cls = max(sizes, key=sizes.get)   # one majority class

    bal_X, bal_y = [], []

    for cls in classes:
        sub = (nn_df[nn_df['Class'] == cls]
               .sort_values(['safe_level', 'distance'], ascending=[False, False])
               .reset_index(drop=True))

        if cls == maj_cls:
            # Undersample: prefer fully-safe examples (same_count == 5)
            safe = sub[sub['same_count'] >= 4]
            if len(safe) >= m:
                chosen_idx = safe.head(m)['real_index'].values
            else:
                rest = sub[sub['same_count'] < 4]
                fill = rest.head(m - len(safe))
                chosen_idx = pd.concat([safe, fill])['real_index'].values
        else:
            # Oversample: sample with replacement, weighted by safe_level
            w = sub['safe_level'].values
            if w.sum() == 0:
                w = np.ones(len(sub))
            w = w / w.sum()
            rows = rng.choice(len(sub), size=m, replace=True, p=w)
            chosen_idx = sub.iloc[rows]['real_index'].values

        bal_X.append(X[chosen_idx])
        bal_y.append(y[chosen_idx])

    X_bal = np.vstack(bal_X)
    y_bal = np.concatenate(bal_y)
    perm  = rng.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

## 4. Evaluation Functions

In [ ]:
def evaluate(clf_name, clf, X_te, y_te, classes):
    y_pred = clf.predict(X_te)
    acc    = metrics.accuracy_score(y_te, y_pred)
    f1     = f1_score(y_te, y_pred, average='macro', zero_division=0)
    gmean  = geometric_mean_score(y_te, y_pred, average='macro')

    try:
        y_prob  = clf.predict_proba(X_te)
        y_bin   = label_binarize(y_te, classes=classes)
        auc_roc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc_roc = float('nan')

    return {'clf': clf_name, 'acc': acc, 'f1': f1, 'gmean': gmean, 'auc': auc_roc}


def train_and_evaluate(X_tr, y_tr, X_te, y_te, classes):
    clfs = {
        'KNN':  KNeighborsClassifier(n_neighbors=5),
        'CART': DecisionTreeClassifier(random_state=0),
        'SVM':  svm.SVC(C=1, probability=True, random_state=0),
    }
    results = []
    for name, clf in clfs.items():
        clf.fit(X_tr, y_tr)
        results.append(evaluate(name, clf, X_te, y_te, classes))
    return results, clfs


def print_results(results):
    print(f'  {"Clf":<6} {"Accuracy":>10} {"F1-Macro":>10} {"G-Mean":>8} {"AUC-ROC":>9}')
    print('  ' + '-' * 47)
    for r in results:
        print(f'  {r["clf"]:<6} {r["acc"]:>10.4f} {r["f1"]:>10.4f} '
              f'{r["gmean"]:>8.4f} {r["auc"]:>9.4f}')

## 5. Full MIRT Pipeline

In [ ]:
def run_mirt(name, X, y, feature_names, seed=42, test_size=0.30):
    print('=' * 65)
    print(f'DATASET: {name}')
    print('=' * 65)

    classes = sorted(np.unique(y).tolist())

    # Step 1 — 5-NN + example types
    print('\n[1] 5-NN neighbourhood & example type classification')
    nn_df = get_5nn_info(X, y)
    print_type_summary(nn_df, classes)

    # Step 2 — Similarity matrix
    print('\n[2] Inter-class similarity matrix')
    mu, vals = compute_similarity_matrix(X, y, seed=seed)
    print(f'  Representative values: { {c: round(v, 4) for c, v in vals.items()} }')
    print_similarity_matrix(mu, classes)

    # Step 3 — Safe levels
    print('\n[3] Safe levels')
    nn_df = compute_safe_levels(nn_df, mu, classes)
    for cls in classes:
        sub = nn_df[nn_df['Class'] == cls]
        print(f'  Class {cls}: mean safe_level = {sub["safe_level"].mean():.4f}')

    # Step 4 — Resampling
    print('\n[4] MIRT Resampling')
    dist_before = dict(pd.Series(y).value_counts().sort_index())
    X_bal, y_bal = mirt_resample(X, y, nn_df, mu, seed=seed)
    dist_after = dict(pd.Series(y_bal).value_counts().sort_index())
    print(f'  Before: {dist_before}  (total={len(y)})')
    print(f'  After:  {dist_after}  (total={len(y_bal)})')

    # Step 5 — Classify
    print('\n[5] Classification (70 / 30 split on balanced data)')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_bal, y_bal, test_size=test_size,
        random_state=seed, stratify=y_bal
    )
    results, fitted_clfs = train_and_evaluate(X_tr, y_tr, X_te, y_te, classes)
    print_results(results)

    return {
        'name':    name,
        'nn_df':   nn_df,
        'mu':      mu,
        'X_orig':  X, 'y_orig': y,
        'X_bal':   X_bal, 'y_bal': y_bal,
        'X_test':  X_te, 'y_test': y_te,
        'results': results,
        'clfs':    fitted_clfs,
        'classes': classes,
    }

## 6. Experiments — All Five Datasets

### 6.1 new-thyroid (NT)

In [ ]:
X_nt, y_nt, feat_nt = load_new_thyroid()
res_nt = run_mirt('new-thyroid (NT)', X_nt, y_nt, feat_nt)

### 6.2 hayes-roth (HR)

In [ ]:
X_hr, y_hr, feat_hr = load_hayes_roth()
res_hr = run_mirt('hayes-roth (HR)', X_hr, y_hr, feat_hr)

### 6.3 balance-scale (BS)

In [ ]:
X_bs, y_bs, feat_bs = load_balance_scale()
res_bs = run_mirt('balance-scale (BS)', X_bs, y_bs, feat_bs)

### 6.4 car (CA)

In [ ]:
X_ca, y_ca, feat_ca = load_car()
res_ca = run_mirt('car (CA)', X_ca, y_ca, feat_ca)

### 6.5 cmc (CMC)

In [ ]:
X_cmc, y_cmc, feat_cmc = load_cmc()
res_cmc = run_mirt('cmc (CMC)', X_cmc, y_cmc, feat_cmc)

## 7. Results Summary

In [ ]:
all_res = [res_nt, res_hr, res_bs, res_ca, res_cmc]

rows = []
for res in all_res:
    for r in res['results']:
        rows.append({
            'Dataset':    res['name'],
            'Classifier': r['clf'],
            'Accuracy':   round(r['acc'],   4),
            'F1-Macro':   round(r['f1'],    4),
            'G-Mean':     round(r['gmean'], 4),
            'AUC-ROC':    round(r['auc'],   4),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 8. Visualisations

### 8.1 Class Distribution — Before vs After Resampling

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 7))
for col, res in enumerate(all_res):
    before = pd.Series(res['y_orig']).value_counts().sort_index()
    after  = pd.Series(res['y_bal']).value_counts().sort_index()
    axes[0, col].bar(before.index.astype(str), before.values, color='steelblue', edgecolor='white')
    axes[0, col].set_title(res['name'], fontsize=8, fontweight='bold')
    if col == 0: axes[0, col].set_ylabel('Before MIRT', fontsize=9)
    axes[1, col].bar(after.index.astype(str), after.values, color='tomato', edgecolor='white')
    if col == 0: axes[1, col].set_ylabel('After MIRT', fontsize=9)
fig.suptitle('Class Distribution Before and After MIRT Resampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mirt_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Accuracy Comparison

In [ ]:
def metric_bar(summary, metric, title, fname):
    pivot = summary.pivot(index='Dataset', columns='Classifier', values=metric)
    ax = pivot.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
    ax.set_ylim(0, 1.08)
    for p in ax.patches:
        h = p.get_height()
        if not np.isnan(h):
            ax.annotate(f'{h:.2f}', (p.get_x() + p.get_width()/2, h),
                        ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

metric_bar(summary, 'Accuracy', 'Accuracy on MIRT-Balanced Datasets', 'mirt_accuracy.png')

### 8.3 F1-Score (Macro)

In [ ]:
metric_bar(summary, 'F1-Macro', 'Macro F1-Score on MIRT-Balanced Datasets', 'mirt_f1.png')

### 8.4 G-Mean

In [ ]:
metric_bar(summary, 'G-Mean', 'G-Mean on MIRT-Balanced Datasets', 'mirt_gmean.png')

### 8.5 AUC ROC

In [ ]:
metric_bar(summary, 'AUC-ROC', 'AUC ROC on MIRT-Balanced Datasets', 'mirt_auc_bar.png')

### 8.6 ROC Curves (hayes-roth — as in paper Figs. 4.3–4.5)

In [ ]:
def plot_roc(res):
    classes  = res['classes']
    X_te     = res['X_test']
    y_te     = res['y_test']
    y_bin    = label_binarize(y_te, classes=classes)
    colors   = ['darkorange', 'royalblue', 'green', 'red']
    n_clfs   = len(res['clfs'])
    fig, axes = plt.subplots(1, n_clfs, figsize=(6 * n_clfs, 5))
    if n_clfs == 1: axes = [axes]
    for ax, (cname, clf) in zip(axes, res['clfs'].items()):
        y_prob = clf.predict_proba(X_te)
        macro_auc = []
        for i, cls in enumerate(classes):
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            v = auc(fpr, tpr)
            macro_auc.append(v)
            ax.plot(fpr, tpr, color=colors[i % len(colors)],
                    label=f'Class {cls} (AUC={v:.2f})')
        ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
        ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
               xlim=[0, 1], ylim=[0, 1.05],
               title=f'{cname} — {res["name"]}\nMacro AUC={np.mean(macro_auc):.2f}')
        ax.legend(fontsize=8)
    plt.tight_layout()
    fname = f'roc_{res["name"].split("(")[1].strip(")")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

plot_roc(res_hr)

In [ ]:
plot_roc(res_nt)

In [ ]:
plot_roc(res_bs)

### 8.7 Confusion Matrices

In [ ]:
def plot_cm(res):
    n = len(res['clfs'])
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1: axes = [axes]
    for ax, (cname, clf) in zip(axes, res['clfs'].items()):
        cm   = confusion_matrix(res['y_test'], clf.predict(res['X_test']))
        disp = ConfusionMatrixDisplay(cm, display_labels=res['classes'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{cname} — {res["name"]}', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'cm_{res["name"].split("(")[1].strip(")")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

for res in all_res:
    plot_cm(res)

### 8.8 Data Difficulty Factor Distribution

In [ ]:
type_rows = []
for res in all_res:
    nn = res['nn_df']
    for t in ['S', 'B', 'R', 'O']:
        type_rows.append({'Dataset': res['name'], 'Type': t,
                          'Pct': 100 * (nn['type'] == t).mean()})
type_df = pd.DataFrame(type_rows)
pivot   = type_df.pivot(index='Dataset', columns='Type', values='Pct')

ax = pivot[['S', 'B', 'R', 'O']].plot(
    kind='bar', stacked=True, figsize=(12, 5),
    color=['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'], edgecolor='white'
)
ax.set_title('Data Difficulty Factor Distribution per Dataset', fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.legend(['Safe (S)', 'Borderline (B)', 'Rare (R)', 'Outlier (O)'])
plt.tight_layout()
plt.savefig('mirt_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Summary Table (Table 4.3 equivalent)

In [ ]:
print('\n╔═══════════════════════════════════════════════════════════════════════╗')
print('║            MIRT — Full Experimental Results Summary                  ║')
print('╠═══════════════════════════════════════════════════════════════════════╣')
print(f'║ {"Dataset":<22} {"Clf":<5} {"Accuracy":>8} {"F1-Macro":>9} {"G-Mean":>8} {"AUC":>7} ║')
print('╠═══════════════════════════════════════════════════════════════════════╣')
prev = ''
for _, row in summary.iterrows():
    ds = row['Dataset'] if row['Dataset'] != prev else ''
    prev = row['Dataset']
    print(f'║ {ds:<22} {row["Classifier"]:<5} '
          f'{row["Accuracy"]:>8.4f} {row["F1-Macro"]:>9.4f} '
          f'{row["G-Mean"]:>8.4f} {row["AUC-ROC"]:>7.4f} ║')
print('╚═══════════════════════════════════════════════════════════════════════╝')

print('\n── Best classifier per dataset (by G-Mean) ─────────────────────────────')
best = summary.loc[summary.groupby('Dataset')['G-Mean'].idxmax()]
print(best[['Dataset', 'Classifier', 'F1-Macro', 'G-Mean', 'AUC-ROC']].to_string(index=False))

## 10. Detailed Classification Reports

In [ ]:
for res in all_res:
    print(f"\n{'='*60}")
    print(f"Dataset: {res['name']}")
    print(f"{'='*60}")
    for cname, clf in res['clfs'].items():
        y_pred = clf.predict(res['X_test'])
        print(f"\n── {cname} ──")
        print(classification_report(res['y_test'], y_pred, digits=3, zero_division=0))